In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("DART_API_KEY")
print(f"API 키 로드: {'✅' if api_key else '❌'}")

API 키 로드: ✅


In [2]:
# 삼성전자 2024년 사업보고서 (연결재무제표) 호출
# corp_code: 00126380 (Day 2에서 매핑한 값)
# bsns_year: 2024 (사업연도)
#  reprt_code: 11011 (사업보고서 = 4분기, 연간)
# 💡 reprt_code 코드표:
# 11011: 사업보고서 (4분기, 연간)
# 11014: 3분기보고서
# 11012: 반기보고서 (2분기)
# 11013: 1분기보고서
# fs_div: CFS (연결재무제표)

url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00126380",
    "bsns_year": "2026",
    "reprt_code": "11013",  # 사업보고서
    "fs_div": "CFS",        # 연결재무제표
}

response = requests.get(url, params=params)
data = response.json()

print(f"응답 상태: {data.get('status')}")
print(f"메시지: {data.get('message')}")
print(f"항목 수: {len(data.get('list', []))}")

응답 상태: 000
메시지: 정상
항목 수: 215


In [9]:
# list 부분을 DataFrame으로 변환
# 💡 핵심 컬럼 의미:
# sj_nm: 재무제표 종류 (손익계산서/재무상태표/현금흐름표)
# account_id: K-IFRS 표준 계정 코드 (영문)
# account_nm: 계정 이름 (한글)
# thstrm_amount: 당기금액 (이번 기간)
# frmtrm_amount: 전기금액 (전년 동기)
df = pd.DataFrame(data["list"])

print(f"전체 행 수: {len(df)}")
print(f"\n=== 컬럼 ===")
print(df.columns.tolist())
print(f"\n=== 재무제표 종류 분포 ===")
print(df["sj_nm"].value_counts())

전체 행 수: 215

=== 컬럼 ===
['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'ord', 'currency', 'thstrm_add_amount', 'frmtrm_q_nm', 'frmtrm_q_amount', 'frmtrm_add_amount']

=== 재무제표 종류 분포 ===
sj_nm
자본변동표      98
재무상태표      49
현금흐름표      38
손익계산서      17
포괄손익계산서    13
Name: count, dtype: int64


In [3]:
# 7개 원천 데이터 검색
targets = {
    "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
    "영업이익": ["dart_OperatingIncomeLoss"],
    "당기순이익": ["ifrs-full_ProfitLoss"],
    "자본총계": ["ifrs-full_Equity"],
    "부채총계": ["ifrs-full_Liabilities"],
    "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
    "감가상각비": ["dart_DepreciationExpense", "dart_DepreciationAndAmortisationExpense"],
}

print("=" * 70)
for name, account_ids in targets.items():
    found = df[df["account_id"].isin(account_ids)]
    if not found.empty:
        for _, row in found.iterrows():
            amount = row["thstrm_amount"]
            sj = row["sj_nm"]
            print(f"✅ {name:8s} | {sj:15s} | {row['account_id']:50s} | {amount}")
    else:
        print(f"❌ {name:8s} | 계정 ID 없음 → 다른 ID 탐색 필요")
print("=" * 70)

NameError: name 'df' is not defined

In [ ]:
# 현금흐름표만 추출
cf = df[df["sj_nm"] == "현금흐름표"]
print(f"현금흐름표 항목 수: {len(cf)}")

# 감가상각 관련 키워드로 검색
print("\n=== '감가' 포함 항목 ===")
depreciation = cf[cf["account_nm"].str.contains("감가|상각|Depreciation|Amortis", na=False, case=False)]
print(depreciation[["account_id", "account_nm", "thstrm_amount"]].to_string())

현금흐름표 항목 수: 38

=== '감가' 포함 항목 ===
Empty DataFrame
Columns: [account_id, account_nm, thstrm_amount]
Index: []


In [5]:
def get_financial_data(corp_code: str, year: int, reprt_code: str, fs_div: str = "CFS") -> dict:
    """
    DART에서 단일 회사의 재무제표를 가져와 7개 원천 데이터를 추출한다.
    
    Args:
        corp_code: DART 기업 고유번호 (8자리)
        year: 사업연도 (예: 2024)
        reprt_code: 보고서 코드
            - 11011: 사업보고서 (4분기, 연간)
            - 11014: 3분기보고서
            - 11012: 반기보고서 (2분기)
            - 11013: 1분기보고서
        fs_div: CFS(연결) 또는 OFS(별도)
    
    Returns:
        dict: {매출액, 영업이익, 당기순이익, 감가상각비, 자본총계, 부채총계, 현금성자산}
              각 값은 정수(원 단위) 또는 None (데이터 없음)
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    
    # 응답 검증
    if data.get("status") != "000":
        return {
            "error": f"DART 응답 오류: {data.get('status')} - {data.get('message')}",
            "corp_code": corp_code,
            "year": year,
            "reprt_code": reprt_code,
        }
    
    df = pd.DataFrame(data["list"])
    
    # 계정 ID 매핑 (회사마다 다를 수 있어 후보 여러 개)
    target_ids = {
        "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
        "영업이익": ["dart_OperatingIncomeLoss"],
        "당기순이익": ["ifrs-full_ProfitLoss"],
        "자본총계": ["ifrs-full_Equity"],
        "부채총계": ["ifrs-full_Liabilities"],
        "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
        "감가상각비": [
            "dart_DepreciationAndAmortisationExpense",
            "dart_DepreciationExpense",
        ],
    }
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    for name, ids in target_ids.items():
        found = df[df["account_id"].isin(ids)]
        if not found.empty:
            # thstrm_amount는 문자열 (콤마 포함) → 정수 변환
            amount_str = found.iloc[0]["thstrm_amount"]
            try:
                # 콤마 제거 + 빈 문자열 처리
                amount = int(amount_str.replace(",", "")) if amount_str else None
            except (ValueError, AttributeError):
                amount = None
            result[name] = amount
        else:
            result[name] = None
    
    return result


# 함수 테스트 — 삼성전자 2024년 사업보고서
samsung_2024 = get_financial_data(
    corp_code="00126380",
    year=2024,
    reprt_code="11011",
    fs_div="CFS",
)

print("=== 삼성전자 2024년 (연결) ===")
for k, v in samsung_2024.items():
    if isinstance(v, int):
        print(f"  {k:12s}: {v:>20,} 원")
    else:
        print(f"  {k:12s}: {v}")

=== 삼성전자 2024년 (연결) ===
  corp_code   : 00126380
  year        :                2,024 원
  reprt_code  : 11011
  fs_div      : CFS
  매출액         :  300,870,903,000,000 원
  영업이익        :   32,725,961,000,000 원
  당기순이익       :   34,451,351,000,000 원
  자본총계        :  402,192,070,000,000 원
  부채총계        :  112,339,878,000,000 원
  현금성자산       :   53,705,579,000,000 원
  감가상각비       : None


In [6]:
# 최근 4분기 데이터 수집
# 2024년: 1Q, 2Q, 3Q, 사업보고서(연간) = 4개 보고서
# 분기별 데이터를 모두 가져온다

quarters = [
    ("2024", "11013", "2024_1Q"),  # 2024 1분기
    ("2024", "11012", "2024_2Q"),  # 2024 반기 (1~2분기 누적)
    ("2024", "11014", "2024_3Q"),  # 2024 3분기 (1~3분기 누적)
    ("2024", "11011", "2024_FY"),  # 2024 사업보고서 (연간)
]

print("4개 보고서 수집 중...\n")
results = []
for year, reprt_code, label in quarters:
    print(f"[{label}] 수집 중...", end=" ")
    data = get_financial_data("00126380", int(year), reprt_code)
    if "error" in data:
        print(f"❌ {data['error']}")
    else:
        print("✅")
    data["label"] = label
    results.append(data)

# DataFrame으로 정리
df_results = pd.DataFrame(results)
display_cols = ["label", "매출액", "영업이익", "당기순이익", "자본총계", "감가상각비"]
print(f"\n=== 분기별 데이터 ===")
print(df_results[display_cols].to_string())

4개 보고서 수집 중...

[2024_1Q] 수집 중... ✅
[2024_2Q] 수집 중... ✅
[2024_3Q] 수집 중... ✅
[2024_FY] 수집 중... ✅

=== 분기별 데이터 ===
     label              매출액            영업이익           당기순이익             자본총계 감가상각비
0  2024_1Q   71915601000000   6606009000000   6754708000000  371916124000000  None
1  2024_2Q   74068302000000  10443878000000   9841345000000  383526671000000  None
2  2024_3Q   79098731000000   9183371000000  10100904000000  386281363000000  None
3  2024_FY  300870903000000  32725961000000  34451351000000  402192070000000  None


In [7]:
# 검증: 우리가 가져온 값이 누적인지 단독인지 확인
# 단서: 손익계산서 항목의 frmtrm_nm, frmtrm_amount (전기 비교)

# 2024 3분기 보고서를 다시 직접 호출해서 원본 응답 확인
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00126380",
    "bsns_year": "2024",
    "reprt_code": "11014",  # 3분기보고서
    "fs_div": "CFS",
}

response = requests.get(url, params=params)
data = response.json()
df_raw = pd.DataFrame(data["list"])

# 매출액 행만 보기
revenue_rows = df_raw[df_raw["account_id"] == "ifrs-full_Revenue"]
print("=== 3분기보고서의 매출액 관련 행 ===")
print(revenue_rows[["sj_nm", "account_nm", "thstrm_nm", "thstrm_amount", "frmtrm_nm", "frmtrm_amount"]].to_string())

=== 3분기보고서의 매출액 관련 행 ===
    sj_nm account_nm   thstrm_nm   thstrm_amount frmtrm_nm frmtrm_amount
67  손익계산서        매출액  제 56 기 3분기  79098731000000       NaN           NaN


In [8]:
def get_financial_data_v5(
    corp_code: str,
    year: int,
    reprt_code: str,
    fs_div: str = "CFS",
) -> dict:
    """
    v5 업데이트:
    - 배당금 추가 (현금흐름표에서 발견)
    - 영업활동현금흐름 → EBITDA 대체
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {"error": f"네트워크 오류: {e}"}
    
    if data.get("status") != "000":
        return {"error": f"DART {data.get('status')}: {data.get('message')}"}
    
    df = pd.DataFrame(data["list"])
    
    target_ids = {
        "매출액": ["ifrs-full_Revenue", "ifrs_Revenue"],
        "영업이익": ["dart_OperatingIncomeLoss"],
        "당기순이익": ["ifrs-full_ProfitLoss"],
        "자본총계": ["ifrs-full_Equity"],
        "부채총계": ["ifrs-full_Liabilities"],
        "현금성자산": ["ifrs-full_CashAndCashEquivalents"],
        "감가상각비": [
            "dart_DepreciationAndAmortisationExpense",
            "dart_DepreciationExpense",
        ],
        # 신규 추가
        "영업활동현금흐름": [
            "ifrs-full_CashFlowsFromUsedInOperatingActivities",
            "dart_CashFlowsFromOperatingActivities",
        ],
        "배당금": [
            "ifrs-full_DividendsPaidClassifiedAsFinancingActivities",
            "ifrs-full_DividendsPaid",
            "dart_DividendsPaid",
        ],
        "자기주식취득": [
            "ifrs-full_PurchaseOfTreasuryShares",
        ],
    }
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "reprt_code": reprt_code,
        "fs_div": fs_div,
    }
    
    for name, ids in target_ids.items():
        found = df[df["account_id"].isin(ids)]
        if not found.empty:
            amount_str = found.iloc[0]["thstrm_amount"]
            try:
                amount = int(amount_str.replace(",", "")) if amount_str and amount_str.strip() else None
                # 배당금/자기주식취득은 음수일 수 있음 → 절대값 사용
                if amount and name in ["배당금", "자기주식취득"]:
                    amount = abs(amount)
                result[name] = amount
            except (ValueError, AttributeError):
                result[name] = None
        else:
            result[name] = None
    
    # EBITDA 계산
    if result["감가상각비"] is not None and result["영업이익"] is not None:
        result["EBITDA"] = result["영업이익"] + result["감가상각비"]
        result["EBITDA_방식"] = "정공법"
    elif result["영업활동현금흐름"] is not None:
        result["EBITDA"] = result["영업활동현금흐름"]
        result["EBITDA_방식"] = "근사치 (영업활동현금흐름)"
    else:
        result["EBITDA"] = None
        result["EBITDA_방식"] = "계산 불가"
    
    return result


# 삼성전자 재테스트
print("=" * 60)
print("삼성전자 재무 데이터 v5 (배당금 + EBITDA 대체)")
print("=" * 60)
fin = get_financial_data_v5("00126380", 2024, "11011")
for k, v in fin.items():
    if isinstance(v, int):
        print(f"  {k:18s}: {v:>20,}")
    else:
        print(f"  {k:18s}: {v}")

삼성전자 재무 데이터 v5 (배당금 + EBITDA 대체)
  corp_code         : 00126380
  year              :                2,024
  reprt_code        : 11011
  fs_div            : CFS
  매출액               :  300,870,903,000,000
  영업이익              :   32,725,961,000,000
  당기순이익             :   34,451,351,000,000
  자본총계              :  402,192,070,000,000
  부채총계              :  112,339,878,000,000
  현금성자산             :   53,705,579,000,000
  감가상각비             : None
  영업활동현금흐름          :   72,982,621,000,000
  배당금               :   10,888,749,000,000
  자기주식취득            :    1,811,775,000,000
  EBITDA            :   72,982,621,000,000
  EBITDA_방식         : 근사치 (영업활동현금흐름)


In [9]:
def get_shares_outstanding_v3(corp_code: str, year: int, reprt_code: str) -> dict:
    """
    v3: 올바른 컬럼 사용 (istc_totqy + distb_stock_co)
    
    핵심 컬럼:
    - istc_totqy: 발행주식의 총수 (실제 발행)
    - tesstk_co: 자기주식수
    - distb_stock_co: 유통주식수 (= istc_totqy - tesstk_co)
    """
    url = "https://opendart.fss.or.kr/api/stockTotqySttus.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
    }
    
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    
    if data.get("status") != "000":
        return {"error": f"DART {data.get('status')}: {data.get('message')}"}
    
    df = pd.DataFrame(data["list"])
    
    def safe_int(val):
        if val is None:
            return 0
        s = str(val).strip()
        if not s or s == "-" or s == "":
            return 0
        try:
            return int(s.replace(",", ""))
        except (ValueError, AttributeError):
            return 0
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "보통주_발행": 0,
        "보통주_자기주식": 0,
        "보통주_유통": 0,
        "우선주_발행": 0,
        "우선주_유통": 0,
    }
    
    for _, row in df.iterrows():
        se = str(row.get("se", "")).strip()
        
        # "보통주" 행만 처리 (합계, 비고 제외)
        if se == "보통주":
            result["보통주_발행"] = safe_int(row.get("istc_totqy"))
            result["보통주_자기주식"] = safe_int(row.get("tesstk_co"))
            result["보통주_유통"] = safe_int(row.get("distb_stock_co"))
        elif se == "우선주":
            result["우선주_발행"] = safe_int(row.get("istc_totqy"))
            result["우선주_유통"] = safe_int(row.get("distb_stock_co"))
        # "합계", "비고" 행은 무시
    
    return result


# 테스트
print("=" * 60)
print("삼성전자 발행주식수 v3 (올바른 컬럼)")
print("=" * 60)
shares = get_shares_outstanding_v3("00126380", 2024, "11011")
for k, v in shares.items():
    if isinstance(v, int):
        print(f"  {k:18s}: {v:>20,} 주")
    else:
        print(f"  {k:18s}: {v}")

print(f"\n--- 검증 ---")
print(f"  보통주_발행 - 자기주식 = {shares['보통주_발행'] - shares['보통주_자기주식']:,}")
print(f"  보통주_유통           = {shares['보통주_유통']:,}")
print(f"  ↑ 두 값이 같으면 OK")

삼성전자 발행주식수 v3 (올바른 컬럼)
  corp_code         : 00126380
  year              :                2,024 주
  보통주_발행            :        5,969,782,550 주
  보통주_자기주식          :           29,700,000 주
  보통주_유통            :        5,940,082,550 주
  우선주_발행            :          822,886,700 주
  우선주_유통            :          818,836,700 주

--- 검증 ---
  보통주_발행 - 자기주식 = 5,940,082,550
  보통주_유통           = 5,940,082,550
  ↑ 두 값이 같으면 OK


In [10]:
def get_current_price(stock_code: str) -> dict:
    """
    종목코드로 최근 종가를 가져온다 (FinanceDataReader 사용).
    """
    try:
        import FinanceDataReader as fdr
        from datetime import datetime, timedelta
        
        # 최근 7일 데이터 조회 (휴장일 대비)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=7)
        df = fdr.DataReader(stock_code, start=start_date, end=end_date)
        
        if df.empty:
            return {"error": "주가 데이터 없음", "stock_code": stock_code}
        
        latest = df.iloc[-1]
        return {
            "stock_code": stock_code,
            "date": df.index[-1].strftime("%Y-%m-%d"),
            "close": int(latest["Close"]),
            "volume": int(latest.get("Volume", 0)),
        }
    except Exception as e:
        return {"error": str(e), "stock_code": stock_code}


# 삼성전자 주가 테스트
print("=== 삼성전자 최근 주가 ===")
price_data = get_current_price("005930")
for k, v in price_data.items():
    if isinstance(v, int):
        print(f"  {k}: {v:>15,}")
    else:
        print(f"  {k}: {v}")

=== 삼성전자 최근 주가 ===
  stock_code: 005930
  date: 2026-06-09
  close:         322,000
  volume:      28,989,253


In [11]:
def calculate_metrics_textbook(
    financials: dict,
    shares_data: dict,
    price: int,
) -> dict:
    """
    교과서 공식으로 10개 지표 계산.
    
    공식:
        EPS = 당기순이익 / 총발행주식수 (보통주 유통)
        BPS = 자본총계 / 총발행주식수
        SPS = 매출액 / 총발행주식수
        ROE = 당기순이익 / 자본총계 × 100
        
    주의: 네이버금융과 차이가 있을 수 있음 (네이버는 지배주주/우선주포함 기준)
    """
    REQUIRED = ["당기순이익", "매출액", "자본총계"]
    missing = [k for k in REQUIRED if financials.get(k) is None]
    if missing:
        return {"error": f"필수 데이터 누락: {missing}"}
    
    shares = shares_data.get("보통주_유통")
    if not shares or not price:
        return {"error": "주식수 또는 주가 없음"}
    
    total_dividend = financials.get("배당금")
    
    # ───────────────────────────────────
    # 교과서 공식 (단순하고 일관됨)
    # ───────────────────────────────────
    # 주당 지표 (4개)
    eps = financials["당기순이익"] / shares
    sps = financials["매출액"] / shares
    bps = financials["자본총계"] / shares
    dps = (total_dividend / shares) if total_dividend else None
    
    # 배당성향
    dividend_payout_ratio = (
        (total_dividend / financials["당기순이익"] * 100)
        if total_dividend else None
    )
    
    # 비율 지표 (5개)
    per = price / eps if eps > 0 else None
    psr = price / sps if sps > 0 else None
    pbr = price / bps if bps > 0 else None
    roe = financials["당기순이익"] / financials["자본총계"] * 100
    
    # EV/EBITDA
    ev_ebitda = None
    market_cap = price * shares
    
    can_calc_ev = (
        financials.get("EBITDA") is not None and
        financials.get("부채총계") is not None and
        financials.get("현금성자산") is not None
    )
    
    if can_calc_ev:
        net_debt = financials["부채총계"] - financials["현금성자산"]
        ev = market_cap + net_debt
        ebitda = financials["EBITDA"]
        ev_ebitda = ev / ebitda if ebitda > 0 else None
    
    return {
        # 10개 지표
        "EPS": round(eps, 2),
        "SPS": round(sps, 2),
        "BPS": round(bps, 2),
        "DPS": round(dps, 2) if dps else None,
        "배당성향(%)": round(dividend_payout_ratio, 2) if dividend_payout_ratio else None,
        "PER": round(per, 2) if per else None,
        "PSR": round(psr, 2) if psr else None,
        "PBR": round(pbr, 2) if pbr else None,
        "ROE(%)": round(roe, 2),
        "EV/EBITDA": round(ev_ebitda, 2) if ev_ebitda else None,
        
        # 메타 (앱에서 작게 표시용)
        "_계산방식": "교과서 공식 (당기순이익/보통주_유통)",
        "_EBITDA_방식": financials.get("EBITDA_방식"),
        "_시가총액": int(market_cap),
        "_주가": price,
        "_보통주_유통": shares,
        "_총배당금": total_dividend,
        
        # 참고용 (지배주주 기준도 같이 저장 — 나중에 옵션화 가능)
        "_참고_지배주주순이익": financials.get("지배주주순이익"),
        "_참고_지배주주자본": financials.get("지배주주자본"),
    }


# 실행
print("=" * 60)
print("📊 삼성전자 — 10개 지표 (교과서 공식)")
print("=" * 60)

# 1. 재무 (v6, 데이터는 풍부하게)
fin = get_financial_data_v5("00126380", 2025, "11011")

# 2. 발행주식수
shares = get_shares_outstanding_v3("00126380", 2025, "11011")

# 3. 주가
price_data = get_current_price("005930")

# 4. 지표 계산 (교과서)
metrics = calculate_metrics_textbook(fin, shares, price_data["close"])

print(f"\n[1/3] 재무: ✅ (당기순이익 {fin['당기순이익']/1e12:.1f}조)")
print(f"[2/3] 보통주 유통: {shares['보통주_유통']:,}주")
print(f"[3/3] 주가: {price_data['close']:,}원")

print("\n" + "=" * 60)
print("🎯 10개 지표 (교과서 공식)")
print("=" * 60)

print("\n📌 주당 지표")
print(f"  EPS : {metrics['EPS']:>12,.2f} 원")
print(f"  SPS : {metrics['SPS']:>12,.2f} 원")
print(f"  BPS : {metrics['BPS']:>12,.2f} 원")
print(f"  DPS : {metrics['DPS']:>12,.2f} 원" if metrics['DPS'] else "  DPS : None")

print("\n📌 배당")
print(f"  배당성향 : {metrics['배당성향(%)']:.2f} %")

print("\n📌 밸류에이션")
print(f"  PER       : {metrics['PER']:>8.2f} 배")
print(f"  PSR       : {metrics['PSR']:>8.2f} 배")
print(f"  PBR       : {metrics['PBR']:>8.2f} 배")
if metrics['EV/EBITDA']:
    print(f"  EV/EBITDA : {metrics['EV/EBITDA']:>8.2f} 배")

print("\n📌 수익성")
print(f"  ROE : {metrics['ROE(%)']:.2f} %")

print(f"\n--- 메타 ---")
print(f"  계산 방식: {metrics['_계산방식']}")
print(f"  주가: {metrics['_주가']:,} 원")
print(f"  시가총액: {metrics['_시가총액'] / 1e12:.1f} 조원")

📊 삼성전자 — 10개 지표 (교과서 공식)

[1/3] 재무: ✅ (당기순이익 45.2조)
[2/3] 보통주 유통: 5,827,808,935주
[3/3] 주가: 322,000원

🎯 10개 지표 (교과서 공식)

📌 주당 지표
  EPS :     7,757.08 원
  SPS :    57,243.80 원
  BPS :    74,868.68 원
  DPS :     1,698.27 원

📌 배당
  배당성향 : 21.89 %

📌 밸류에이션
  PER       :    41.51 배
  PSR       :     5.63 배
  PBR       :     4.30 배
  EV/EBITDA :    22.85 배

📌 수익성
  ROE : 10.36 %

--- 메타 ---
  계산 방식: 교과서 공식 (당기순이익/보통주_유통)
  주가: 322,000 원
  시가총액: 1876.6 조원


In [12]:
print("=" * 60)
print("📊 삼성전자 — 10개 지표 (수정판)")
print("=" * 60)

# 1. 재무 데이터 (이미 메모리에 있을 듯)
fin = get_financial_data_v5("00126380", 2024, "11011")
print(f"\n[1/3] 재무 데이터: ✅")

# 2. 발행주식수 (v3 함수!)
shares = get_shares_outstanding_v3("00126380", 2024, "11011")
print(f"[2/3] 보통주 유통: {shares['보통주_유통']:,}주")

# 3. 주가
price_data = get_current_price("005930")
print(f"[3/3] 주가: {price_data['close']:,}원 ({price_data['date']})")

# 4. 10개 지표
metrics = calculate_metrics_textbook(fin, shares, price_data["close"])

print("\n" + "=" * 60)
print("🎯 10개 지표")
print("=" * 60)

print("\n📌 주당 지표")
print(f"  EPS : {metrics['EPS']:>12,.2f} 원")
print(f"  SPS : {metrics['SPS']:>12,.2f} 원")
print(f"  BPS : {metrics['BPS']:>12,.2f} 원")
print(f"  DPS : {metrics['DPS']:>12,.2f} 원" if metrics['DPS'] else "  DPS : None")

print("\n📌 배당")
print(f"  배당성향 : {metrics['배당성향(%)']:.2f} %")

print("\n📌 밸류에이션")
print(f"  PER       : {metrics['PER']:>8.2f} 배")
print(f"  PSR       : {metrics['PSR']:>8.2f} 배")
print(f"  PBR       : {metrics['PBR']:>8.2f} 배")
print(f"  EV/EBITDA : {metrics['EV/EBITDA']:>8.2f} 배")

print("\n📌 수익성")
print(f"  ROE : {metrics['ROE(%)']:.2f} %")

print(f"\n--- 메타 ---")
print(f"  주가: {metrics['_주가']:,} 원")
print(f"  시가총액: {metrics['_시가총액'] / 1e12:.1f} 조원")
print(f"  EBITDA 방식: {metrics['_EBITDA_방식']}")

📊 삼성전자 — 10개 지표 (수정판)

[1/3] 재무 데이터: ✅
[2/3] 보통주 유통: 5,940,082,550주
[3/3] 주가: 322,000원 (2026-06-09)

🎯 10개 지표

📌 주당 지표
  EPS :     5,799.81 원
  SPS :    50,650.96 원
  BPS :    67,708.16 원
  DPS :     1,833.10 원

📌 배당
  배당성향 : 31.61 %

📌 밸류에이션
  PER       :    55.52 배
  PSR       :     6.36 배
  PBR       :     4.76 배
  EV/EBITDA :    27.01 배

📌 수익성
  ROE : 8.57 %

--- 메타 ---
  주가: 322,000 원
  시가총액: 1912.7 조원
  EBITDA 방식: 근사치 (영업활동현금흐름)


In [13]:
def analyze_company(name: str, corp_code: str, stock_code: str, 
                    year: int = 2025, reprt_code: str = "11011") -> dict:
    """
    한 회사의 전 과정을 한 번에:
    재무 → 발행주식수 → 주가 → 10개 지표 계산
    """
    print(f"\n[{name}] 분석 중...", end=" ")
    
    # 1. 재무 데이터
    fin = get_financial_data_v5(corp_code, year, reprt_code)
    if "error" in fin:
        print(f"❌ 재무 데이터: {fin['error']}")
        return {"회사명": name, "error": fin["error"]}
    
    # 2. 발행주식수
    shares = get_shares_outstanding_v3(corp_code, year, reprt_code)
    if "error" in shares or not shares.get("보통주_유통"):
        print(f"❌ 발행주식수")
        return {"회사명": name, "error": "발행주식수 없음"}
    
    # 3. 주가
    price_data = get_current_price(stock_code)
    if "error" in price_data:
        print(f"❌ 주가: {price_data['error']}")
        return {"회사명": name, "error": price_data["error"]}
    
    # 4. 지표 계산
    metrics = calculate_metrics_textbook(fin, shares, price_data["close"])
    if "error" in metrics:
        print(f"❌ 지표 계산: {metrics['error']}")
        return {"회사명": name, "error": metrics["error"]}
    
    print(f"✅ (주가 {price_data['close']:,}원)")
    
    return {
        "회사명": name,
        "종목코드": stock_code,
        "주가": price_data["close"],
        "EPS": metrics["EPS"],
        "SPS": metrics["SPS"],
        "BPS": metrics["BPS"],
        "DPS": metrics["DPS"],
        "배당성향(%)": metrics["배당성향(%)"],
        "PER": metrics["PER"],
        "PSR": metrics["PSR"],
        "PBR": metrics["PBR"],
        "ROE(%)": metrics["ROE(%)"],
        "EV/EBITDA": metrics["EV/EBITDA"],
        "시가총액(조)": round(metrics["_시가총액"] / 1e12, 1),
        "EBITDA방식": metrics["_EBITDA_방식"],
    }


# 5종목 정의
test_companies = [
    {"name": "삼성전자",    "corp_code": "00126380", "stock_code": "005930"},
    {"name": "SK하이닉스", "corp_code": "00164779", "stock_code": "000660"},
    {"name": "카카오",      "corp_code": "00258801", "stock_code": "035720"},
    {"name": "셀트리온",    "corp_code": "00413046", "stock_code": "068270"},
    {"name": "KB금융",      "corp_code": "00688996", "stock_code": "105560"},
]

# 일괄 분석
print("=" * 70)
print("Day 5 - 5종목 일괄 분석 (2024년 사업보고서 기준)")
print("=" * 70)

results = []
for company in test_companies:
    result = analyze_company(**company)
    results.append(result)

print("\n" + "=" * 70)
print("✅ 모든 종목 분석 완료")
print("=" * 70)

Day 5 - 5종목 일괄 분석 (2024년 사업보고서 기준)

[삼성전자] 분석 중... ✅ (주가 322,000원)

[SK하이닉스] 분석 중... ✅ (주가 2,215,000원)

[카카오] 분석 중... ✅ (주가 39,500원)

[셀트리온] 분석 중... ❌ 발행주식수

[KB금융] 분석 중... ❌ 지표 계산: 필수 데이터 누락: ['매출액']

✅ 모든 종목 분석 완료


In [14]:
# 성공/실패 분리
success = [r for r in results if "error" not in r]
failed = [r for r in results if "error" in r]

# 표 출력
print("\n=== 📊 5종목 종합 결과 ===\n")

if success:
    df = pd.DataFrame(success)
    
    # 보기 좋은 컬럼 순서
    cols = ["회사명", "주가", "EPS", "BPS", "PER", "PBR", "ROE(%)", "배당성향(%)", "시가총액(조)"]
    print(df[cols].to_string(index=False))
    
    print(f"\n총 {len(success)}개 종목 성공")

if failed:
    print("\n=== ❌ 실패한 종목 ===")
    for r in failed:
        print(f"  - {r['회사명']}: {r['error']}")


=== 📊 5종목 종합 결과 ===

   회사명      주가      EPS       BPS   PER   PBR  ROE(%)  배당성향(%)  시가총액(조)
  삼성전자  322000  7757.08  74868.68 41.51  4.30   10.36    21.89   1876.6
SK하이닉스 2215000 61206.24 171965.53 36.19 12.88   35.59     3.91   1554.2
   카카오   39500  1177.04  34597.81 33.56  1.14    3.40     7.52     17.4

총 3개 종목 성공

=== ❌ 실패한 종목 ===
  - 셀트리온: 발행주식수 없음
  - KB금융: 필수 데이터 누락: ['매출액']


In [15]:
print("=" * 70)
print("삼성전자 2025년 사업보고서 (가장 최신 연간 데이터)")
print("=" * 70)

fin_2025 = get_financial_data_v5("00126380", 2025, "11011")

if "error" in fin_2025:
    print(f"❌ 에러: {fin_2025['error']}")
else:
    for k, v in fin_2025.items():
        if isinstance(v, int):
            print(f"  {k:18s}: {v:>22,}")
        else:
            print(f"  {k:18s}: {v}")

삼성전자 2025년 사업보고서 (가장 최신 연간 데이터)
  corp_code         : 00126380
  year              :                  2,025
  reprt_code        : 11011
  fs_div            : CFS
  매출액               :    333,605,938,000,000
  영업이익              :     43,601,051,000,000
  당기순이익             :     45,206,805,000,000
  자본총계              :    436,320,337,000,000
  부채총계              :    130,621,773,000,000
  현금성자산             :     57,856,378,000,000
  감가상각비             : None
  영업활동현금흐름          :     85,315,148,000,000
  배당금               :      9,897,183,000,000
  자기주식취득            :      8,189,263,000,000
  EBITDA            :     85,315,148,000,000
  EBITDA_방식         : 근사치 (영업활동현금흐름)


In [16]:
shares_2025 = get_shares_outstanding_v3("00126380", 2025, "11011")

print("=" * 60)
print("삼성전자 발행주식수 (2025년 사업보고서 기준)")
print("=" * 60)
for k, v in shares_2025.items():
    if isinstance(v, int):
        print(f"  {k}: {v:>20,} 주")
    else:
        print(f"  {k}: {v}")

삼성전자 발행주식수 (2025년 사업보고서 기준)
  corp_code: 00126380
  year:                2,025 주
  보통주_발행:        5,919,637,922 주
  보통주_자기주식:           91,828,987 주
  보통주_유통:        5,827,808,935 주
  우선주_발행:          815,974,664 주
  우선주_유통:          802,371,203 주


In [18]:
print("=" * 60)
print("📊 삼성전자 — 10개 지표 (2025년 사업보고서 + 2026년 6월 주가)")
print("=" * 60)

# 1. 재무 (2025년)
fin = get_financial_data_v5("00126380", 2025, "11011")
print(f"\n[1/3] 재무: ✅ 매출 {fin['매출액'] / 1e12:.1f}조")

# 2. 발행주식수 (2025년)
shares = get_shares_outstanding_v3("00126380", 2025, "11011")
print(f"[2/3] 보통주 유통: {shares['보통주_유통']:,}주")

# 3. 주가 (현재)
price_data = get_current_price("005930")
print(f"[3/3] 주가: {price_data['close']:,}원 ({price_data['date']})")

# 4. 지표 계산
metrics = calculate_metrics_textbook(fin, shares, price_data["close"])

print("\n" + "=" * 60)
print("🎯 10개 지표 (2025년 기준)")
print("=" * 60)

print("\n📌 주당 지표")
print(f"  EPS : {metrics['EPS']:>12,.2f} 원")
print(f"  SPS : {metrics['SPS']:>12,.2f} 원")
print(f"  BPS : {metrics['BPS']:>12,.2f} 원")
print(f"  DPS : {metrics['DPS']:>12,.2f} 원" if metrics['DPS'] else "  DPS : None")

print("\n📌 배당")
if metrics['배당성향(%)']:
    print(f"  배당성향 : {metrics['배당성향(%)']:.2f} %")

print("\n📌 밸류에이션")
print(f"  PER       : {metrics['PER']:>8.2f} 배")
print(f"  PSR       : {metrics['PSR']:>8.2f} 배")
print(f"  PBR       : {metrics['PBR']:>8.2f} 배")
if metrics['EV/EBITDA']:
    print(f"  EV/EBITDA : {metrics['EV/EBITDA']:>8.2f} 배")

print("\n📌 수익성")
print(f"  ROE : {metrics['ROE(%)']:.2f} %")

print(f"\n--- 메타 ---")
print(f"  주가: {metrics['_주가']:,} 원")
print(f"  시가총액: {metrics['_시가총액'] / 1e12:.1f} 조원")

📊 삼성전자 — 10개 지표 (2025년 사업보고서 + 2026년 6월 주가)

[1/3] 재무: ✅ 매출 333.6조
[2/3] 보통주 유통: 5,827,808,935주
[3/3] 주가: 322,000원 (2026-06-09)

🎯 10개 지표 (2025년 기준)

📌 주당 지표
  EPS :     7,757.08 원
  SPS :    57,243.80 원
  BPS :    74,868.68 원
  DPS :     1,698.27 원

📌 배당
  배당성향 : 21.89 %

📌 밸류에이션
  PER       :    41.51 배
  PSR       :     5.63 배
  PBR       :     4.30 배
  EV/EBITDA :    22.85 배

📌 수익성
  ROE : 10.36 %

--- 메타 ---
  주가: 322,000 원
  시가총액: 1876.6 조원


In [21]:
# 셀트리온 발행주식수 API 원본 응답 확인
url = "https://opendart.fss.or.kr/api/stockTotqySttus.json"
params = {
    "crtfc_key": api_key,
    "corp_code": "00413046",  # 셀트리온
    "bsns_year": "2025",
    "reprt_code": "11011",
}

response = requests.get(url, params=params)
data = response.json()

print("=" * 70)
print("셀트리온 발행주식수 API 진단")
print("=" * 70)
print(f"응답 상태: {data.get('status')}")
print(f"메시지: {data.get('message')}")
print(f"\nlist 길이: {len(data.get('list', []))}")

if data.get('list'):
    df = pd.DataFrame(data['list'])
    print(f"\n=== 전체 응답 ===")
    print(df.to_string())
    
    print(f"\n=== se(증권종류) 값 ===")
    print(df["se"].unique())
else:
    print("\n⚠️ 빈 응답. 2025년 데이터 없을 가능성.")
    print("2024년으로 재시도...")
    
    params["bsns_year"] = "2024"
    response2 = requests.get(url, params=params)
    data2 = response2.json()
    print(f"\n2024년 응답 상태: {data2.get('status')}")
    if data2.get('list'):
        df2 = pd.DataFrame(data2['list'])
        print(df2.to_string())

셀트리온 발행주식수 API 진단
응답 상태: 000
메시지: 정상

list 길이: 4

=== 전체 응답 ===
         rcept_no corp_cls corp_code corp_name         se isu_stock_totqy now_to_isu_stock_totqy now_to_dcrs_stock_totqy       redc profit_incnr rdmstk_repy         etc   istc_totqy   tesstk_co distb_stock_co     stlm_dt
0  20260316001415        Y  00413046      셀트리온  의결권 있는 주식     400,000,000            331,580,746             100,619,777  4,819,244    8,400,687           -  87,399,846  230,960,969  12,347,845    218,613,124  2025-12-31
1  20260316001415        Y  00413046      셀트리온  의결권 없는 주식               -                      -                       -          -            -           -           -            -           -              -  2025-12-31
2  20260316001415        Y  00413046      셀트리온         합계     400,000,000            331,580,746             100,619,777  4,819,244    8,400,687           -  87,399,846  230,960,969  12,347,845    218,613,124  2025-12-31
3  20260316001415        Y  00413046      셀트리온      

In [22]:
def get_shares_outstanding_v4(corp_code: str, year: int, reprt_code: str) -> dict:
    """
    v4: 두 가지 증권 표기 방식 모두 지원
    - "보통주" 또는 "의결권 있는 주식" → 보통주로 처리
    - "우선주" 또는 "의결권 없는 주식" → 우선주로 처리
    """
    url = "https://opendart.fss.or.kr/api/stockTotqySttus.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": reprt_code,
    }
    
    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    
    if data.get("status") != "000":
        return {"error": f"DART {data.get('status')}: {data.get('message')}"}
    
    df = pd.DataFrame(data["list"])
    
    def safe_int(val):
        if val is None:
            return 0
        s = str(val).strip()
        if not s or s == "-" or s == "":
            return 0
        try:
            return int(s.replace(",", ""))
        except (ValueError, AttributeError):
            return 0
    
    result = {
        "corp_code": corp_code,
        "year": year,
        "보통주_발행": 0,
        "보통주_자기주식": 0,
        "보통주_유통": 0,
        "우선주_발행": 0,
        "우선주_유통": 0,
        "증권표기방식": "",  # 디버깅용
    }
    
    for _, row in df.iterrows():
        se = str(row.get("se", "")).strip()
        
        # 보통주 (또는 의결권 있는 주식)
        if se == "보통주" or se == "의결권 있는 주식":
            result["보통주_발행"] = safe_int(row.get("istc_totqy"))
            result["보통주_자기주식"] = safe_int(row.get("tesstk_co"))
            result["보통주_유통"] = safe_int(row.get("distb_stock_co"))
            result["증권표기방식"] = se
        
        # 우선주 (또는 의결권 없는 주식)
        elif se == "우선주" or se == "의결권 없는 주식":
            result["우선주_발행"] = safe_int(row.get("istc_totqy"))
            result["우선주_유통"] = safe_int(row.get("distb_stock_co"))
        
        # "합계", "비고" 행은 무시
    
    return result


# 셀트리온 테스트
print("=" * 60)
print("셀트리온 발행주식수 v4 (의결권 표기 지원)")
print("=" * 60)
shares = get_shares_outstanding_v4("00413046", 2025, "11011")
for k, v in shares.items():
    if isinstance(v, int):
        print(f"  {k:20s}: {v:>20,} 주")
    else:
        print(f"  {k:20s}: {v}")


# 삼성전자도 정상 작동하는지 확인
print("\n" + "=" * 60)
print("삼성전자 발행주식수 v4 (회귀 테스트)")
print("=" * 60)
shares_samsung = get_shares_outstanding_v4("00126380", 2025, "11011")
for k, v in shares_samsung.items():
    if isinstance(v, int):
        print(f"  {k:20s}: {v:>20,} 주")
    else:
        print(f"  {k:20s}: {v}")

셀트리온 발행주식수 v4 (의결권 표기 지원)
  corp_code           : 00413046
  year                :                2,025 주
  보통주_발행              :          230,960,969 주
  보통주_자기주식            :           12,347,845 주
  보통주_유통              :          218,613,124 주
  우선주_발행              :                    0 주
  우선주_유통              :                    0 주
  증권표기방식              : 의결권 있는 주식

삼성전자 발행주식수 v4 (회귀 테스트)
  corp_code           : 00126380
  year                :                2,025 주
  보통주_발행              :        5,919,637,922 주
  보통주_자기주식            :           91,828,987 주
  보통주_유통              :        5,827,808,935 주
  우선주_발행              :          815,974,664 주
  우선주_유통              :          802,371,203 주
  증권표기방식              : 보통주


In [24]:
def analyze_company_v3(
    name: str,
    corp_code: str,
    stock_code: str,
    year: int = 2025,
    reprt_code: str = "11011",
) -> dict:
    """v3: 발행주식수 v4 (의결권 표기) + 적자 처리"""
    print(f"\n[{name}] 분석 중...", end=" ")
    
    fin = get_financial_data_v5(corp_code, year, reprt_code)
    if "error" in fin:
        print(f"❌ 재무: {fin['error']}")
        return {"회사명": name, "error": fin["error"]}
    
    # ⭐ v4 사용
    shares = get_shares_outstanding_v4(corp_code, year, reprt_code)
    if "error" in shares or not shares.get("보통주_유통"):
        print(f"❌ 발행주식수")
        return {"회사명": name, "error": "발행주식수 없음"}
    
    price_data = get_current_price(stock_code)
    if "error" in price_data:
        print(f"❌ 주가: {price_data['error']}")
        return {"회사명": name, "error": price_data["error"]}
    
    # ⭐ v2 (적자 처리) 사용 - 미리 만든 calculate_metrics_v2가 있어야 함
    # 없으면 기존 calculate_metrics 사용
    try:
        metrics = calculate_metrics_textbook(fin, shares, price_data["close"])
    except NameError:
        metrics = calculate_metrics_textbook(fin, shares, price_data["close"])
    
    if "error" in metrics:
        print(f"❌ 지표: {metrics['error']}")
        return {"회사명": name, "error": metrics["error"]}
    
    print(f"✅ (주가 {price_data['close']:,}원)")
    
    return {
        "회사명": name,
        "종목코드": stock_code,
        "주가": price_data["close"],
        "EPS": metrics["EPS"],
        "BPS": metrics["BPS"],
        "DPS": metrics["DPS"],
        "PER": metrics["PER"],
        "PBR": metrics["PBR"],
        "ROE(%)": metrics["ROE(%)"],
        "배당성향(%)": metrics["배당성향(%)"],
        "EV/EBITDA": metrics["EV/EBITDA"],
        "시가총액(조)": round(metrics["_시가총액"] / 1e12, 1),
    }


# 5종목 재실행
test_companies = [
    {"name": "삼성전자",    "corp_code": "00126380", "stock_code": "005930"},
    {"name": "SK하이닉스", "corp_code": "00164779", "stock_code": "000660"},
    {"name": "카카오",      "corp_code": "00258801", "stock_code": "035720"},
    {"name": "셀트리온",    "corp_code": "00413046", "stock_code": "068270"},
    {"name": "KB금융",      "corp_code": "00688996", "stock_code": "105560"},
]

print("=" * 70)
print("🎯 Day 5 FINAL - 5종목 일괄 분석 v3 (셀트리온 + 적자처리)")
print("=" * 70)

results = []
for company in test_companies:
    result = analyze_company_v3(**company, year=2025)
    results.append(result)

# 결과 정리
success = [r for r in results if "error" not in r]
failed = [r for r in results if "error" in r]

print("\n" + "=" * 70)
print(f"📊 결과: 성공 {len(success)} / 실패 {len(failed)}")
print("=" * 70)

if success:
    df = pd.DataFrame(success)
    cols = ["회사명", "주가", "EPS", "BPS", "PER", "PBR", "ROE(%)", "배당성향(%)", "시가총액(조)"]
    print(df[cols].to_string(index=False))

if failed:
    print("\n=== ❌ 실패 ===")
    for r in failed:
        print(f"  - {r['회사명']}: {r['error']}")

🎯 Day 5 FINAL - 5종목 일괄 분석 v3 (셀트리온 + 적자처리)

[삼성전자] 분석 중... ✅ (주가 322,000원)

[SK하이닉스] 분석 중... ✅ (주가 2,215,000원)

[카카오] 분석 중... ✅ (주가 39,500원)

[셀트리온] 분석 중... ✅ (주가 170,000원)

[KB금융] 분석 중... ❌ 지표: 필수 데이터 누락: ['매출액']

📊 결과: 성공 4 / 실패 1
   회사명      주가      EPS       BPS   PER   PBR  ROE(%)  배당성향(%)  시가총액(조)
  삼성전자  322000  7757.08  74868.68 41.51  4.30   10.36    21.89   1876.6
SK하이닉스 2215000 61206.24 171965.53 36.19 12.88   35.59     3.91   1554.2
   카카오   39500  1177.04  34597.81 33.56  1.14    3.40     7.52     17.4
  셀트리온  170000  4718.31  79375.52 36.03  2.14    5.94    14.91     37.2

=== ❌ 실패 ===
  - KB금융: 필수 데이터 누락: ['매출액']
